# NLI Incremental Dataset Combination Evaluation: mDeBERTa-v3-base-mnli-xnli

Evaluates **MoritzLaurer/mDeBERTa-v3-base-mnli-xnli** as a **zero-shot base model** across **all 7 possible dataset combinations** to measure the **incremental contribution** of each dataset.

## Experiment Logic

| Scenario | Datasets Evaluated ON | Type |
|---|---|---|
| S1 | SNLI-TR only | Single |
| S2 | MultiNLI-TR only | Single |
| S3 | TrGLUE only | Single |
| S1+S2 | SNLI-TR + MultiNLI-TR | Pair |
| S1+S3 | SNLI-TR + TrGLUE | Pair |
| S2+S3 | MultiNLI-TR + TrGLUE | Pair |
| S1+S2+S3 | All three datasets | Triple |

## Key Question

> **Which dataset, when added to an existing combination, contributes the most to overall performance?**

## How This Differs from Qwen Notebooks

| Property | Qwen2-7B / 1.5B | mDeBERTa-v3-base-mnli-xnli |
|---|---|---|
| Architecture | Generative LLM | Encoder-only classifier |
| Inference method | Prompted text generation | Sequence-pair classification |
| NLI head | None (zero-shot prompt) | Pre-trained 3-class head |
| Parameters | 1.5B / 7B | ~184M |
| Speed | Slow (generative) | Fast (encoder, BATCH_SIZE=32) |
| Expected accuracy | ~50-65% | ~75-85% (fine-tuned on XNLI)

**Merging strategy:** `concatenate_datasets()` in memory — originals on HuggingFace are **NOT modified**.

**Metrics:** Accuracy, Macro F1, Weighted F1, Per-class F1, Confusion Matrix (CSV + heatmap)

## 1. Install Dependencies

In [1]:
!pip install -q -U transformers datasets accelerate scikit-learn tqdm huggingface_hub[hf_transfer] seaborn matplotlib pandas

## 2. Imports & Environment Setup

In [2]:
import json
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    classification_report,
)
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False
    print('matplotlib/seaborn not available — PNG heatmaps will be skipped.')

# Enable faster HuggingFace downloads
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Device detection: CUDA (Windows/Linux) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    print('Apple Silicon MPS available; using for acceleration.')
else:
    print('No GPU/MPS detected — running on CPU.')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


No GPU/MPS detected — running on CPU.


## 3. Configuration

In [3]:
# Model
MODEL_ID   = 'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
BATCH_SIZE = 32   # encoder models are fast; lower to 8-16 if OOM

# Dataset hub repo
REPO_ID = 'yilmazzey/sdp2-nli'
CONFIGS  = ['snli_tr_1_1', 'multinli_tr_1_1', 'trglue_mnli']

# Eval splits per dataset — test/validation ONLY, train is never touched
EVAL_SPLITS = {
    'snli_tr_1_1'     : ['test'],
    'multinli_tr_1_1' : ['validation_matched', 'validation_mismatched'],
    'trglue_mnli'     : ['test_matched', 'test_mismatched'],
}

# All 7 non-empty subsets — singles, pairs, triple
SCENARIOS = {
    'single_snli'       : ['snli_tr_1_1'],
    'single_mnli'       : ['multinli_tr_1_1'],
    'single_trglue'     : ['trglue_mnli'],
    'pair_snli_mnli'    : ['snli_tr_1_1',     'multinli_tr_1_1'],
    'pair_snli_trglue'  : ['snli_tr_1_1',     'trglue_mnli'],
    'pair_mnli_trglue'  : ['multinli_tr_1_1', 'trglue_mnli'],
    'triple_all'        : ['snli_tr_1_1',     'multinli_tr_1_1', 'trglue_mnli'],
}

# Output root
RESULTS_DIR = 'results combined'

# Evaluation settings
SAMPLE_LIMIT = 200   # None = whole dataset; set e.g. 200 for quick test

LABEL_NAMES = ['entailment', 'neutral', 'contradiction']
NUM_LABELS  = 3


## 4. Load All Dataset Configs

Loaded **read-only** from HuggingFace into local cache — originals are **never modified**.
Only `test` / `validation` splits are used; `train` is never touched.

In [4]:
all_datasets = {}
for cfg in CONFIGS:
    print(f'Loading {REPO_ID} :: {cfg} ...')
    all_datasets[cfg] = load_dataset(REPO_ID, cfg)
    print('  available splits:', list(all_datasets[cfg].keys()))
    print('  eval splits used:', EVAL_SPLITS[cfg])
    for sp in EVAL_SPLITS[cfg]:
        if sp in all_datasets[cfg]:
            print(f'    {sp}: {len(all_datasets[cfg][sp])} rows')


Loading yilmazzey/sdp2-nli :: snli_tr_1_1 ...
  available splits: ['train', 'validation', 'test']
  eval splits used: ['test']
    test: 9824 rows
Loading yilmazzey/sdp2-nli :: multinli_tr_1_1 ...


  available splits: ['train', 'validation_matched', 'validation_mismatched']
  eval splits used: ['validation_matched', 'validation_mismatched']
    validation_matched: 9809 rows
    validation_mismatched: 9825 rows
Loading yilmazzey/sdp2-nli :: trglue_mnli ...
  available splits: ['train', 'validation_matched', 'validation_mismatched', 'test_matched', 'test_mismatched']
  eval splits used: ['test_matched', 'test_mismatched']
    test_matched: 9008 rows
    test_mismatched: 9217 rows


## 5. Load Model & Tokenizer

mDeBERTa uses `AutoModelForSequenceClassification` — **not** a text-generation pipeline.
The model already has a pre-trained 3-class NLI head (entailment / neutral / contradiction).
No prompting needed; inference is a simple forward pass.

In [5]:
print(f'Loading {MODEL_ID} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    ignore_mismatched_sizes=True,
)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

model = model.to(device)
model.eval()
print(f'Using device: {device}')
print('Model loaded successfully.')
print(f'Model label order: {model.config.id2label}')


Loading MoritzLaurer/mDeBERTa-v3-base-mnli-xnli ...


config.json: 0.00B [00:00, ?B/s]

C:\Users\Pc\PycharmProjects\SDP2\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pc\.cache\huggingface\hub\models--MoritzLaurer--mDeBERTa-v3-base-mnli-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cpu
Model loaded successfully.
Model label order: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


## 6. Helper Functions

**Key difference from Qwen:** mDeBERTa uses **sequence-pair classification**:
- Tokenize `(premise, hypothesis)` together with `[SEP]`
- Forward pass → logits → `argmax` → predicted label
- No text generation, no prompt parsing needed

**Label mapping** is taken directly from `model.config.id2label` to ensure correctness.
Typical order for this model: `{0: entailment, 1: neutral, 2: contradiction}`.

In [6]:
def tokenize_fn(examples):
    """Tokenize premise+hypothesis pairs for sequence classification."""
    return tokenizer(
        examples['premise'],
        examples['hypothesis'],
        truncation=True,
        max_length=256,
    )


def run_inference(split_data, n=None):
    """Batched sequence-pair classification inference."""
    if n is not None:
        split_data = split_data.select(range(min(n, len(split_data))))

    # Tokenize — keep only 'label' from original columns
    remove_cols = [c for c in split_data.column_names if c != 'label']
    tokenized   = split_data.map(
        tokenize_fn,
        batched=True,
        remove_columns=remove_cols,
        desc='Tokenize',
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def collate_fn(examples):
        labels = torch.tensor([ex['label'] for ex in examples])
        batch  = data_collator([{k: v for k, v in ex.items() if k != 'label'} for ex in examples])
        batch['labels'] = labels
        return batch

    loader = torch.utils.data.DataLoader(
        tokenized, batch_size=BATCH_SIZE, collate_fn=collate_fn
    )
    preds_list, labels_list = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Inference'):
            out = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
            )
            preds_list.append(out.logits.argmax(-1).cpu().numpy())
            labels_list.append(batch['labels'].numpy())

    y_pred = np.concatenate(preds_list)
    y_true = np.concatenate(labels_list)
    return y_true.tolist(), y_pred.tolist()


def compute_metrics(y_true, y_pred):
    """Returns full metrics dict + confusion matrix."""
    acc         = float(accuracy_score(y_true, y_pred))
    f1_macro    = float(f1_score(y_true, y_pred, average='macro',    zero_division=0))
    f1_weighted = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))
    f1_per      = f1_score(y_true, y_pred, average=None,             zero_division=0)
    f1_per_cls  = {LABEL_NAMES[i]: float(f1_per[i]) for i in range(NUM_LABELS)}
    cm          = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cr          = classification_report(y_true, y_pred, target_names=LABEL_NAMES,
                                        zero_division=0, output_dict=True)
    return {
        'accuracy'              : acc,
        'f1_macro'              : f1_macro,
        'f1_weighted'           : f1_weighted,
        'f1_per_class'          : f1_per_cls,
        'classification_report' : cr,
    }, cm


def save_confusion(cm, out_dir, name):
    """Save confusion matrix as CSV + optional PNG heatmap."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / f'confusion_{name}.csv'
    np.savetxt(csv_path, cm, fmt='%d', delimiter=',',
               header=','.join(LABEL_NAMES), comments='')
    print(f'  Confusion CSV  -> {csv_path}')
    if HAS_PLOT:
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d',
                    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(name)
        plt.tight_layout()
        png_path = out_dir / f'confusion_{name}.png'
        plt.savefig(png_path); plt.close()
        print(f'  Confusion PNG  -> {png_path}')


def collect_splits(cfg_list):
    """Returns list of (cfg_name, split_name, split_data) tuples."""
    result = []
    for cfg in cfg_list:
        for sp in EVAL_SPLITS[cfg]:
            if sp in all_datasets[cfg]:
                result.append((cfg, sp, all_datasets[cfg][sp]))
            else:
                print(f'  Warning: {cfg}/{sp} not found, skipping.')
    return result


def run_scenario(scenario_name, cfg_list):
    """
    Evaluate mDeBERTa on all eval splits of every dataset in cfg_list.
    Saves per-split metrics, a merged combined score, confusion matrices,
    and a metrics.json under results combined/<scenario_name>/.

    Args:
        scenario_name : folder name, e.g. 'pair_snli_mnli'
        cfg_list      : list of dataset config names to evaluate on
    Returns:
        dict with full results
    """
    out_dir = Path(RESULTS_DIR) / scenario_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print('\n' + '='*65)
    print(f'SCENARIO : {scenario_name}')
    print(f'Datasets : {cfg_list}')
    print('='*65)

    splits         = collect_splits(cfg_list)
    per_split_m    = {}
    all_y_true, all_y_pred = [], []

    for cfg, sp, split_data in splits:
        key = f'{cfg}__{sp}'
        print(f'\n  Evaluating {cfg} / {sp}  ({len(split_data)} rows) ...')
        y_true, y_pred = run_inference(split_data, n=SAMPLE_LIMIT)
        print('    True dist:', dict(Counter(y_true)))
        print('    Pred dist:', dict(Counter(y_pred)))
        metrics, cm = compute_metrics(y_true, y_pred)
        per_split_m[key] = metrics
        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)
        print(f"    acc={metrics['accuracy']:.4f}  f1_macro={metrics['f1_macro']:.4f}  f1_weighted={metrics['f1_weighted']:.4f}")
        save_confusion(cm, out_dir, key)

    print(f'\n  Combined over {len(splits)} split(s) ...')
    combined_m, combined_cm = compute_metrics(all_y_true, all_y_pred)
    print(f"  COMBINED acc={combined_m['accuracy']:.4f}  f1_macro={combined_m['f1_macro']:.4f}  f1_weighted={combined_m['f1_weighted']:.4f}")
    save_confusion(combined_cm, out_dir, f'{scenario_name}_combined')

    result = {
        'model'     : MODEL_ID,
        'scenario'  : scenario_name,
        'datasets'  : cfg_list,
        'n_splits'  : len(splits),
        'per_split' : per_split_m,
        'combined'  : combined_m,
    }
    with open(out_dir / 'metrics.json', 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"\n  Saved -> {out_dir / 'metrics.json'}")
    return result


## 7. Single Dataset Scenarios

Baseline: each dataset evaluated **individually**.
These scores are the reference points for measuring incremental gain.

In [7]:
result_single_snli = run_scenario(
    scenario_name = 'single_snli',
    cfg_list      = ['snli_tr_1_1']
)



SCENARIO : single_snli
Datasets : ['snli_tr_1_1']

  Evaluating snli_tr_1_1 / test  (9824 rows) ...


Tokenize:   0%|          | 0/200 [00:00<?, ? examples/s]

Inference: 100%|██████████| 7/7 [07:10<00:00, 61.48s/it]


    True dist: {1: 65, 0: 69, 2: 66}
    Pred dist: {1: 68, 0: 90, 2: 42}
    acc=0.7350  f1_macro=0.7273  f1_weighted=0.7287
  Confusion CSV  -> results combined\single_snli\confusion_snli_tr_1_1__test.csv
  Confusion PNG  -> results combined\single_snli\confusion_snli_tr_1_1__test.png

  Combined over 1 split(s) ...
  COMBINED acc=0.7350  f1_macro=0.7273  f1_weighted=0.7287
  Confusion CSV  -> results combined\single_snli\confusion_single_snli_combined.csv
  Confusion PNG  -> results combined\single_snli\confusion_single_snli_combined.png

  Saved -> results combined\single_snli\metrics.json


In [8]:
result_single_mnli = run_scenario(
    scenario_name = 'single_mnli',
    cfg_list      = ['multinli_tr_1_1']
)



SCENARIO : single_mnli
Datasets : ['multinli_tr_1_1']

  Evaluating multinli_tr_1_1 / validation_matched  (9809 rows) ...


Tokenize:   0%|          | 0/200 [00:00<?, ? examples/s]

Inference: 100%|██████████| 7/7 [16:06<00:00, 138.04s/it]


    True dist: {1: 59, 2: 59, 0: 82}
    Pred dist: {1: 64, 2: 56, 0: 80}
    acc=0.7650  f1_macro=0.7657  f1_weighted=0.7657
  Confusion CSV  -> results combined\single_mnli\confusion_multinli_tr_1_1__validation_matched.csv
  Confusion PNG  -> results combined\single_mnli\confusion_multinli_tr_1_1__validation_matched.png

  Evaluating multinli_tr_1_1 / validation_mismatched  (9825 rows) ...


Tokenize:   0%|          | 0/200 [00:00<?, ? examples/s]

Inference: 100%|██████████| 7/7 [16:12<00:00, 138.98s/it]


    True dist: {2: 66, 0: 65, 1: 69}
    Pred dist: {2: 66, 0: 73, 1: 61}
    acc=0.7800  f1_macro=0.7800  f1_weighted=0.7794
  Confusion CSV  -> results combined\single_mnli\confusion_multinli_tr_1_1__validation_mismatched.csv
  Confusion PNG  -> results combined\single_mnli\confusion_multinli_tr_1_1__validation_mismatched.png

  Combined over 2 split(s) ...
  COMBINED acc=0.7725  f1_macro=0.7732  f1_weighted=0.7726
  Confusion CSV  -> results combined\single_mnli\confusion_single_mnli_combined.csv
  Confusion PNG  -> results combined\single_mnli\confusion_single_mnli_combined.png

  Saved -> results combined\single_mnli\metrics.json


In [9]:
result_single_trglue = run_scenario(
    scenario_name = 'single_trglue',
    cfg_list      = ['trglue_mnli']
)



SCENARIO : single_trglue
Datasets : ['trglue_mnli']

  Evaluating trglue_mnli / test_matched  (9008 rows) ...


Tokenize:   0%|          | 0/200 [00:00<?, ? examples/s]

Inference: 100%|██████████| 7/7 [12:19<00:00, 105.60s/it]


    True dist: {1: 74, 2: 59, 0: 67}
    Pred dist: {1: 52, 0: 85, 2: 63}
    acc=0.8100  f1_macro=0.8070  f1_weighted=0.8043
  Confusion CSV  -> results combined\single_trglue\confusion_trglue_mnli__test_matched.csv
  Confusion PNG  -> results combined\single_trglue\confusion_trglue_mnli__test_matched.png

  Evaluating trglue_mnli / test_mismatched  (9217 rows) ...


Tokenize:   0%|          | 0/200 [00:00<?, ? examples/s]

Inference: 100%|██████████| 7/7 [12:18<00:00, 105.43s/it]


    True dist: {1: 65, 0: 73, 2: 62}
    Pred dist: {1: 51, 0: 80, 2: 69}
    acc=0.8550  f1_macro=0.8514  f1_weighted=0.8528
  Confusion CSV  -> results combined\single_trglue\confusion_trglue_mnli__test_mismatched.csv
  Confusion PNG  -> results combined\single_trglue\confusion_trglue_mnli__test_mismatched.png

  Combined over 2 split(s) ...
  COMBINED acc=0.8325  f1_macro=0.8290  f1_weighted=0.8287
  Confusion CSV  -> results combined\single_trglue\confusion_single_trglue_combined.csv
  Confusion PNG  -> results combined\single_trglue\confusion_single_trglue_combined.png

  Saved -> results combined\single_trglue\metrics.json


## 8. Pair Dataset Scenarios

All three possible two-dataset combinations.
Comparing each pair score to its two single scores reveals the **incremental gain** from combining.

In [10]:
result_pair_snli_mnli = run_scenario(
    scenario_name = 'pair_snli_mnli',
    cfg_list      = ['snli_tr_1_1', 'multinli_tr_1_1']
)



SCENARIO : pair_snli_mnli
Datasets : ['snli_tr_1_1', 'multinli_tr_1_1']

  Evaluating snli_tr_1_1 / test  (9824 rows) ...


Tokenize:   0%|          | 0/200 [00:00<?, ? examples/s]

Inference: 100%|██████████| 7/7 [06:57<00:00, 59.68s/it]


    True dist: {1: 65, 0: 69, 2: 66}
    Pred dist: {1: 68, 0: 90, 2: 42}
    acc=0.7350  f1_macro=0.7273  f1_weighted=0.7287
  Confusion CSV  -> results combined\pair_snli_mnli\confusion_snli_tr_1_1__test.csv
  Confusion PNG  -> results combined\pair_snli_mnli\confusion_snli_tr_1_1__test.png

  Evaluating multinli_tr_1_1 / validation_matched  (9809 rows) ...


Inference: 100%|██████████| 7/7 [15:46<00:00, 135.28s/it]


    True dist: {1: 59, 2: 59, 0: 82}
    Pred dist: {1: 64, 2: 56, 0: 80}
    acc=0.7650  f1_macro=0.7657  f1_weighted=0.7657
  Confusion CSV  -> results combined\pair_snli_mnli\confusion_multinli_tr_1_1__validation_matched.csv
  Confusion PNG  -> results combined\pair_snli_mnli\confusion_multinli_tr_1_1__validation_matched.png

  Evaluating multinli_tr_1_1 / validation_mismatched  (9825 rows) ...


Inference: 100%|██████████| 7/7 [14:12<00:00, 121.73s/it]


    True dist: {2: 66, 0: 65, 1: 69}
    Pred dist: {2: 66, 0: 73, 1: 61}
    acc=0.7800  f1_macro=0.7800  f1_weighted=0.7794
  Confusion CSV  -> results combined\pair_snli_mnli\confusion_multinli_tr_1_1__validation_mismatched.csv
  Confusion PNG  -> results combined\pair_snli_mnli\confusion_multinli_tr_1_1__validation_mismatched.png

  Combined over 3 split(s) ...
  COMBINED acc=0.7600  f1_macro=0.7592  f1_weighted=0.7599
  Confusion CSV  -> results combined\pair_snli_mnli\confusion_pair_snli_mnli_combined.csv
  Confusion PNG  -> results combined\pair_snli_mnli\confusion_pair_snli_mnli_combined.png

  Saved -> results combined\pair_snli_mnli\metrics.json


In [11]:
result_pair_snli_trglue = run_scenario(
    scenario_name = 'pair_snli_trglue',
    cfg_list      = ['snli_tr_1_1', 'trglue_mnli']
)



SCENARIO : pair_snli_trglue
Datasets : ['snli_tr_1_1', 'trglue_mnli']

  Evaluating snli_tr_1_1 / test  (9824 rows) ...


Inference: 100%|██████████| 7/7 [06:55<00:00, 59.42s/it]


    True dist: {1: 65, 0: 69, 2: 66}
    Pred dist: {1: 68, 0: 90, 2: 42}
    acc=0.7350  f1_macro=0.7273  f1_weighted=0.7287
  Confusion CSV  -> results combined\pair_snli_trglue\confusion_snli_tr_1_1__test.csv
  Confusion PNG  -> results combined\pair_snli_trglue\confusion_snli_tr_1_1__test.png

  Evaluating trglue_mnli / test_matched  (9008 rows) ...


Inference: 100%|██████████| 7/7 [12:54<00:00, 110.68s/it]


    True dist: {1: 74, 2: 59, 0: 67}
    Pred dist: {1: 52, 0: 85, 2: 63}
    acc=0.8100  f1_macro=0.8070  f1_weighted=0.8043
  Confusion CSV  -> results combined\pair_snli_trglue\confusion_trglue_mnli__test_matched.csv
  Confusion PNG  -> results combined\pair_snli_trglue\confusion_trglue_mnli__test_matched.png

  Evaluating trglue_mnli / test_mismatched  (9217 rows) ...


Inference: 100%|██████████| 7/7 [12:39<00:00, 108.51s/it]


    True dist: {1: 65, 0: 73, 2: 62}
    Pred dist: {1: 51, 0: 80, 2: 69}
    acc=0.8550  f1_macro=0.8514  f1_weighted=0.8528
  Confusion CSV  -> results combined\pair_snli_trglue\confusion_trglue_mnli__test_mismatched.csv
  Confusion PNG  -> results combined\pair_snli_trglue\confusion_trglue_mnli__test_mismatched.png

  Combined over 3 split(s) ...
  COMBINED acc=0.8000  f1_macro=0.7960  f1_weighted=0.7966
  Confusion CSV  -> results combined\pair_snli_trglue\confusion_pair_snli_trglue_combined.csv
  Confusion PNG  -> results combined\pair_snli_trglue\confusion_pair_snli_trglue_combined.png

  Saved -> results combined\pair_snli_trglue\metrics.json


In [12]:
result_pair_mnli_trglue = run_scenario(
    scenario_name = 'pair_mnli_trglue',
    cfg_list      = ['multinli_tr_1_1', 'trglue_mnli']
)



SCENARIO : pair_mnli_trglue
Datasets : ['multinli_tr_1_1', 'trglue_mnli']

  Evaluating multinli_tr_1_1 / validation_matched  (9809 rows) ...


Inference: 100%|██████████| 7/7 [17:26<00:00, 149.56s/it]


    True dist: {1: 59, 2: 59, 0: 82}
    Pred dist: {1: 64, 2: 56, 0: 80}
    acc=0.7650  f1_macro=0.7657  f1_weighted=0.7657
  Confusion CSV  -> results combined\pair_mnli_trglue\confusion_multinli_tr_1_1__validation_matched.csv
  Confusion PNG  -> results combined\pair_mnli_trglue\confusion_multinli_tr_1_1__validation_matched.png

  Evaluating multinli_tr_1_1 / validation_mismatched  (9825 rows) ...


Inference: 100%|██████████| 7/7 [16:25<00:00, 140.78s/it]


    True dist: {2: 66, 0: 65, 1: 69}
    Pred dist: {2: 66, 0: 73, 1: 61}
    acc=0.7800  f1_macro=0.7800  f1_weighted=0.7794
  Confusion CSV  -> results combined\pair_mnli_trglue\confusion_multinli_tr_1_1__validation_mismatched.csv
  Confusion PNG  -> results combined\pair_mnli_trglue\confusion_multinli_tr_1_1__validation_mismatched.png

  Evaluating trglue_mnli / test_matched  (9008 rows) ...


Inference: 100%|██████████| 7/7 [12:30<00:00, 107.18s/it]


    True dist: {1: 74, 2: 59, 0: 67}
    Pred dist: {1: 52, 0: 85, 2: 63}
    acc=0.8100  f1_macro=0.8070  f1_weighted=0.8043
  Confusion CSV  -> results combined\pair_mnli_trglue\confusion_trglue_mnli__test_matched.csv
  Confusion PNG  -> results combined\pair_mnli_trglue\confusion_trglue_mnli__test_matched.png

  Evaluating trglue_mnli / test_mismatched  (9217 rows) ...


Inference: 100%|██████████| 7/7 [12:21<00:00, 105.97s/it]


    True dist: {1: 65, 0: 73, 2: 62}
    Pred dist: {1: 51, 0: 80, 2: 69}
    acc=0.8550  f1_macro=0.8514  f1_weighted=0.8528
  Confusion CSV  -> results combined\pair_mnli_trglue\confusion_trglue_mnli__test_mismatched.csv
  Confusion PNG  -> results combined\pair_mnli_trglue\confusion_trglue_mnli__test_mismatched.png

  Combined over 4 split(s) ...
  COMBINED acc=0.8025  f1_macro=0.8011  f1_weighted=0.8009
  Confusion CSV  -> results combined\pair_mnli_trglue\confusion_pair_mnli_trglue_combined.csv
  Confusion PNG  -> results combined\pair_mnli_trglue\confusion_pair_mnli_trglue_combined.png

  Saved -> results combined\pair_mnli_trglue\metrics.json


## 9. Triple Dataset Scenario (All Combined)

All three datasets merged together — the upper-bound of dataset coverage.

In [13]:
result_triple_all = run_scenario(
    scenario_name = 'triple_all',
    cfg_list      = ['snli_tr_1_1', 'multinli_tr_1_1', 'trglue_mnli']
)



SCENARIO : triple_all
Datasets : ['snli_tr_1_1', 'multinli_tr_1_1', 'trglue_mnli']

  Evaluating snli_tr_1_1 / test  (9824 rows) ...


Inference: 100%|██████████| 7/7 [06:45<00:00, 57.94s/it]


    True dist: {1: 65, 0: 69, 2: 66}
    Pred dist: {1: 68, 0: 90, 2: 42}
    acc=0.7350  f1_macro=0.7273  f1_weighted=0.7287
  Confusion CSV  -> results combined\triple_all\confusion_snli_tr_1_1__test.csv
  Confusion PNG  -> results combined\triple_all\confusion_snli_tr_1_1__test.png

  Evaluating multinli_tr_1_1 / validation_matched  (9809 rows) ...


Inference: 100%|██████████| 7/7 [17:20<00:00, 148.59s/it]


    True dist: {1: 59, 2: 59, 0: 82}
    Pred dist: {1: 64, 2: 56, 0: 80}
    acc=0.7650  f1_macro=0.7657  f1_weighted=0.7657
  Confusion CSV  -> results combined\triple_all\confusion_multinli_tr_1_1__validation_matched.csv
  Confusion PNG  -> results combined\triple_all\confusion_multinli_tr_1_1__validation_matched.png

  Evaluating multinli_tr_1_1 / validation_mismatched  (9825 rows) ...


...

    True dist: {2: 66, 0: 65, 1: 69}
    Pred dist: {2: 66, 0: 73, 1: 61}
    acc=0.7800  f1_macro=0.7800  f1_weighted=0.7794
  Confusion CSV  -> results combined\triple_all\confusion_multinli_tr_1_1__validation_mismatched.csv
  Confusion PNG  -> results combined\triple_all\confusion_multinli_tr_1_1__validation_mismatched.png

  Evaluating trglue_mnli / test_matched  (9008 rows) ...


Inference: 100%|██████████| 7/7 [11:45<00:00, 100.79s/it]


    True dist: {1: 74, 2: 59, 0: 67}
    Pred dist: {1: 52, 0: 85, 2: 63}
    acc=0.8100  f1_macro=0.8070  f1_weighted=0.8043
  Confusion CSV  -> results combined\triple_all\confusion_trglue_mnli__test_matched.csv
  Confusion PNG  -> results combined\triple_all\confusion_trglue_mnli__test_matched.png

  Evaluating trglue_mnli / test_mismatched  (9217 rows) ...


Inference: 100%|██████████| 7/7 [11:59<00:00, 102.72s/it]


    True dist: {1: 65, 0: 73, 2: 62}
    Pred dist: {1: 51, 0: 80, 2: 69}
    acc=0.8550  f1_macro=0.8514  f1_weighted=0.8528
  Confusion CSV  -> results combined\triple_all\confusion_trglue_mnli__test_mismatched.csv
  Confusion PNG  -> results combined\triple_all\confusion_trglue_mnli__test_mismatched.png

  Combined over 5 split(s) ...
  COMBINED acc=0.7890  f1_macro=0.7871  f1_weighted=0.7875
  Confusion CSV  -> results combined\triple_all\confusion_triple_all_combined.csv
  Confusion PNG  -> results combined\triple_all\confusion_triple_all_combined.png

  Saved -> results combined\triple_all\metrics.json


## 10. Final Summary — Incremental Contribution Analysis

Builds three outputs:
1. **All 7 scenarios ranked** by F1 Macro
2. **Incremental gain table** — pair vs. average of its two singles; triple vs. each pair
3. **Key findings** — best scenario, best pair gain, which dataset adds the most

Saves master `results incremental/metrics.json`.

In [14]:
all_results = {
    'single_snli'       : result_single_snli,
    'single_mnli'       : result_single_mnli,
    'single_trglue'     : result_single_trglue,
    'pair_snli_mnli'    : result_pair_snli_mnli,
    'pair_snli_trglue'  : result_pair_snli_trglue,
    'pair_mnli_trglue'  : result_pair_mnli_trglue,
    'triple_all'        : result_triple_all,
}

# ── Table 1: all scenarios ranked by F1 Macro ───────────────────────────
rows = []
for name, res in all_results.items():
    m    = res['combined']
    n_ds = len(res['datasets'])
    rows.append({
        'Scenario'         : name,
        'Type'             : 'single' if n_ds == 1 else ('pair' if n_ds == 2 else 'triple'),
        'Datasets'         : ' + '.join(res['datasets']),
        'Accuracy'         : round(m['accuracy'],    4),
        'F1 Macro'         : round(m['f1_macro'],    4),
        'F1 Weighted'      : round(m['f1_weighted'], 4),
        'F1 Entailment'    : round(m['f1_per_class']['entailment'],    4),
        'F1 Neutral'       : round(m['f1_per_class']['neutral'],       4),
        'F1 Contradiction' : round(m['f1_per_class']['contradiction'], 4),
    })

summary_df = pd.DataFrame(rows).set_index('Scenario').sort_values('F1 Macro', ascending=False)
print('=' * 80)
print(f'ALL SCENARIOS RANKED BY F1 MACRO  |  Model: {MODEL_ID}')
print('=' * 80)
print(summary_df.to_string())

# ── Table 2: incremental gain ────────────────────────────────────────────
pair_map = {
    'pair_snli_mnli'   : ('single_snli',   'single_mnli',   'trglue_mnli'),
    'pair_snli_trglue' : ('single_snli',   'single_trglue', 'multinli_tr_1_1'),
    'pair_mnli_trglue' : ('single_mnli',   'single_trglue', 'snli_tr_1_1'),
}
gain_rows = []
triple_f1 = all_results['triple_all']['combined']['f1_macro']
for pair_name, (s1, s2, missing) in pair_map.items():
    pair_f1  = all_results[pair_name]['combined']['f1_macro']
    avg_s_f1 = (all_results[s1]['combined']['f1_macro'] +
                all_results[s2]['combined']['f1_macro']) / 2
    gain_rows.append({
        'Pair'                : pair_name,
        'Missing Dataset'     : missing,
        'Avg Single F1'       : round(avg_s_f1,            4),
        'Pair F1 Macro'       : round(pair_f1,             4),
        'Gain vs Avg Single'  : round(pair_f1 - avg_s_f1,  4),
        'Triple F1 Macro'     : round(triple_f1,           4),
        'Gain Pair -> Triple' : round(triple_f1 - pair_f1, 4),
    })

gain_df = pd.DataFrame(gain_rows).set_index('Pair')
print('\n' + '=' * 80)
print('INCREMENTAL GAIN ANALYSIS')
print('=' * 80)
print(gain_df.to_string())

# ── Key findings ────────────────────────────────────────────────────────
print('\n' + '=' * 80)
print('KEY FINDINGS')
print('=' * 80)
print(f'  Best  scenario        : {summary_df.index[0]}  (F1 Macro = {summary_df.iloc[0]["F1 Macro"]:.4f})')
print(f'  Worst scenario        : {summary_df.index[-1]}  (F1 Macro = {summary_df.iloc[-1]["F1 Macro"]:.4f})')
best_pair = gain_df['Gain vs Avg Single'].idxmax()
print(f'  Best pair gain        : {best_pair}  (+{gain_df.loc[best_pair, "Gain vs Avg Single"]:.4f} over avg single)')
best_boost = gain_df['Gain Pair -> Triple'].idxmax()
print(f'  Pair gaining most from 3rd dataset : {best_boost}  (missing: {gain_df.loc[best_boost, "Missing Dataset"]})')

# ── Save master metrics.json ─────────────────────────────────────────────
master = Path(RESULTS_DIR) / 'metrics.json'
master.parent.mkdir(parents=True, exist_ok=True)
with open(master, 'w', encoding='utf-8') as f:
    json.dump({
        'model'      : MODEL_ID,
        'experiment' : 'incremental_dataset_combination_evaluation',
        'scenarios'  : all_results,
    }, f, indent=2, ensure_ascii=False)
print(f'\nMaster metrics saved -> {master}')


ALL SCENARIOS RANKED BY F1 MACRO  |  Model: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
                    Type                                     Datasets  Accuracy  F1 Macro  F1 Weighted  F1 Entailment  F1 Neutral  F1 Contradiction
Scenario                                                                                                                                           
single_trglue     single                                  trglue_mnli    0.8325    0.8290       0.8287         0.8721      0.7769            0.8379
pair_mnli_trglue    pair                multinli_tr_1_1 + trglue_mnli    0.8025    0.8011       0.8009         0.8198      0.7556            0.8280
pair_snli_trglue    pair                    snli_tr_1_1 + trglue_mnli    0.8000    0.7960       0.7966         0.8491      0.7467            0.7922
triple_all        triple  snli_tr_1_1 + multinli_tr_1_1 + trglue_mnli    0.7890    0.7871       0.7875         0.8168      0.7420            0.8026
single_mnli       single    